In [ ]:
import os
import copy
import ipynbname
import pandas as pd
import numpy as np
from Functions.AutoCloud_V4_6 import *
from Functions.Utils import *
from Functions.Graphs import *
from Functions.TedaGraphs import *
from Functions.Utils_OPT_R2 import *
from Functions.TedaOptimize_V4_8 import *     # Optimize, OptimizeSeq, BestParams, SimSeq
pd.set_option('display.float_format', '{:.3f}'.format)
pd.set_option('display.expand_frame_repr', False) # never wrap into a second block

FileName = ipynbname.name()

RS = pd.read_csv(r'Dataset\DSIR1n\B1_1.csv')
HI = pd.read_csv(r'Dataset\HI1\Bearing1_1.csv')
sig = HI['PC1'].values
PARAMS_PATH = r'Optimization\TedaOpt_R6_000_B1_1-B1_2_restart_symmetric_G0P0\mono\Optimization.csv'
df = pd.read_csv(PARAMS_PATH).iloc[:,:]
dfs = df.sort_values(df.columns[-4], ascending=True)
EOL = 0.3

In [4]:
print(dfs)

    SCORE_mean     m  nI                nR  nO  mO  N1  N2  N3  TAU  activation   w  WAPE_RUL_B1_1  WAPE_HI_B1_1  WAPE_RUL_B1_2  WAPE_HI_B1_2
39       1.028 1.762  37  [57, 55, 38, 68]   8  25   4   8   7    8           0   4          0.211         0.160          0.967         0.718
9        0.348 1.950  19              [45]   8   1   2   2   5   13           0  10          0.280         0.055          0.306         0.054
33       1.831 1.962  39   [28, 1, 38, 59]   6  21   1   6   6    8           0   9          0.554         0.015          3.086         0.006
6        1.405 1.925  20              [74]   1   6   9   9   1   43           1   2          0.855         0.351          0.930         0.674
27       1.670 1.837  31      [26, 10, 25]   3  17   5   1   7    3           0   8          0.929         0.653          0.951         0.807
36       1.712 1.938  39  [71, 13, 65, 71]   3  23   3   3   5   47           1   7          0.949         0.865          0.966         0.644
16    

In [3]:
i = 34
params = df.iloc[i, 1:12].values
m,nI,nR,nO,mO,N1,N2,N3,τ,act,W = ParseParams(params)
RHO = 0.1       # rho estático: intervalo do HI = predição -/+ RHO * erro estimado (protocolo 3)
X, Y, Z, kX = PrepareData(RS, HI, W, nI, nO, mO)
ref, J, L = RefEoL(sig, EOL, kX) 

teda = AutoCloud(m=m,nI=nI,nR=nR,nO=nO,mO=mO,ηS=[N1,N2,N3],
			  tau=τ,eol=EOL,ref=ref,act=act,
			  wtaG=False,wtaP=False,store=True,feedback='symmetric',
			  rho=RHO,intr_state='carry'
			  )

teda.SimGranularRUL(J,L,X,Y,Z,'single')
teda.HI_Granularity()
print(f"HI : cov={teda.cov:.3f} | sp={teda.sp:.3f} | V=média(cov*sp)={teda.V:.4f} | Vsum={teda.Vsum:.4f} | ")

PlotSimulation(teda=teda,mode='ply',xi=10, yf=100,ftrs=5,dsi_mode='hide')

HI : cov=0.000 | sp=1.000 | V=média(cov*sp)=0.0000 | Vsum=0.0000 | 


In [9]:
RS1 = pd.read_csv(r'Dataset\DSIR1n\B1_1.csv')
HI = pd.read_csv(r'Dataset\HI1\Bearing1_1.csv')
i = 9
sig = HI['PC1'].values
params = df.iloc[i, 1:12].values
m,nI,nR,nO,mO,N1,N2,N3,τ,act,W = ParseParams(params)
RHO = 0.1       # rho estático: intervalo do HI = predição -/+ RHO * erro estimado (protocolo 3)
X, Y, Z, kX = PrepareData(RS1, HI, W, nI, nO, mO)
ref, J, L = RefEoL(sig, EOL, kX) 
teda = AutoCloud(m=m,nI=nI,nR=nR,nO=nO,mO=mO,ηS=[N1,N2,N3],
			  tau=τ,eol=EOL,ref=ref,act=act,
			  wtaG=False,wtaP=False,store=True,feedback='symmetric',
			  rho=RHO,intr_state='carry'
			  )

teda.SimGranularRUL(J,L,X,Y,Z,'single')
print(f"HI : cov={teda.cov:.3f} | sp={teda.sp:.3f} | V=média(cov*sp)={teda.V:.4f} | Vsum={teda.Vsum:.4f} | ")

PlotSimulation(teda=teda,mode='ply',xi=10, yf=100,ftrs=5,dsi_mode='hide')

RS2 = pd.read_csv(r'Dataset\DSIR1n\B1_2.csv')
HI = pd.read_csv(r'Dataset\HI1\Bearing1_2.csv')
sig = HI['PC1'].values
X, Y, Z, kX = PrepareData(RS2, HI, W, nI, nO, mO)
ref, J, L = RefEoL(sig, EOL, kX) 

teda.restart(ref,'keep',Y[0])
teda.SimGranularRUL(J,L,X,Y,Z,'single')
print(f"HI : cov={teda.cov:.3f} | sp={teda.sp:.3f} | V=média(cov*sp)={teda.V:.4f} | Vsum={teda.Vsum:.4f} | ")
PlotSimulation(teda=teda,mode='ply',xi=10, yf=100,ftrs=5,dsi_mode='hide')

HI : cov=0.000 | sp=1.000 | V=média(cov*sp)=0.0000 | Vsum=0.0000 | 


HI : cov=0.000 | sp=1.000 | V=média(cov*sp)=0.0000 | Vsum=0.0000 | 


In [ ]:
brngs = [f'B1_{i+1}' for i in range(5)]
brngs = ['B1_1','B1_2','B1_3','B1_4','B1_5','B2_1','B2_2','B2_4',][:2]
paths = [f'Dataset\DSIR1n\{brng}.csv' for brng in brngs]
DSIs = [pd.read_csv(path) for path in paths]
#DSIs = [DSI.drop(["FTF"], axis=1) for DSI in DSIs]
ftrs = len(DSIs[0].columns)
DSIs = [RS1,RS2]
teda = AutoCloud(m=m,nI=nI,nR=nR,nO=nO,mO=mO,ηS=[N1,N2,N3],
			  tau=τ,eol=EOL,ref=ref,act=act,
			  wtaG=False,wtaP=False,store=True,feedback='symmetric',
			  rho=RHO,intr_state='carry'
			  )
w = 1
for DSI in DSIs: 
    teda.SimGranulation(WindowDSI(DSI[19:],w))

print(f"HI (HI_Granularity): cov={teda.cov:.3f} | sp={teda.sp:.3f} | V=média(cov*sp)={teda.V:.4f} | Vsum={teda.Vsum:.4f}")
PlotSimulation(teda=teda,mode='ply',xi=10, yf=100,ftrs=ftrs,dsi_mode='hide')

In [ ]:
RS1 = pd.read_csv(r'Dataset\DSIR1n\B1_1.csv')
HI = pd.read_csv(r'Dataset\HI1\Bearing1_1.csv')

sig = HI['PC1'].values
PARAMS_PATH = r'Optimization\TedaRNN_Opt_R3_Seq_B1_1-B1_2_restart_symmetric_G0P0\mono\Optimization.csv'
dfP = pd.read_csv(PARAMS_PATH).iloc[:,:11]
params = dfP.sort_values(dfP.columns[0], ascending=True).iloc[0, 1:].values
m,nI,nR,nO,mO,N1,N2,N3,τ,act = ParseParams(params)
RHO = 0.1       # rho estático: intervalo do HI = predição -/+ RHO * erro estimado (protocolo 3)
X1, Y, Z, kX = PrepareData(RS1, HI, W, nI, nO, mO)
ref, J, L = RefEoL(sig, EOL, kX) 
teda = AutoCloud(m=m,nI=nI,nR=nR,nO=nO,mO=mO,ηS=[N1,N2,N3],
			  tau=τ,eol=EOL,ref=ref,act=act,
			  wtaG=False,wtaP=False,store=True,feedback='symmetric',
			  rho=RHO,intr_state='carry'
			  )

teda.SimGranularRUL(J,L,X1,Y,Z,'single')
print(f"HI : cov={teda.cov:.3f} | sp={teda.sp:.3f} | V=média(cov*sp)={teda.V:.4f} | Vsum={teda.Vsum:.4f} | ")

PlotSimulation(teda=teda,mode='ply',xi=10, yf=100,ftrs=5,dsi_mode='hide')

RS2 = pd.read_csv(r'Dataset\DSIR1n\B1_2.csv')
HI = pd.read_csv(r'Dataset\HI1\Bearing1_2.csv')
sig = HI['PC1'].values
X2, Y, Z, kX = PrepareData(RS2, HI, W, nI, nO, mO)
ref, J, L = RefEoL(sig, EOL, kX) 

teda.restart(ref,'keep',Y[0])
teda.SimGranularRUL(J,L,X2,Y,Z,'single')
print(f"HI : cov={teda.cov:.3f} | sp={teda.sp:.3f} | V=média(cov*sp)={teda.V:.4f} | Vsum={teda.Vsum:.4f} | ")
PlotSimulation(teda=teda,mode='ply',xi=10, yf=100,ftrs=5,dsi_mode='hide')

In [ ]:
wR = (0.101690 - 0)/(0.3 - 0) 
wH = (0.013605	 - 0)/(0.2 - 0) 
print(wR, wH, wR+wH, wR*wH)

wR = (0.575803	 - 0)/(0.3 - 0) 
wH = (0.003675	 - 0)/(0.2 - 0) 
print(wR, wH, wR+wH, wR*wH)

In [ ]:
wR = (0.226344 - 0)/(0.3 - 0) 
wH = (0.001195		 - 0)/(0.2 - 0) 
print(wR, wH,wR+wH, wR*wH)

wR = (0.224894 - 0)/(0.3 - 0) 
wH = (0.000693	 - 0)/(0.2 - 0) 
print(wR, wH,wR+wH, wR*wH)

In [ ]:
path = r'Optimization\TedaOpt_R5_T0_B1_1-B1_2_restart_symmetric_G0P0\mono\Optimization.csv'
df = pd.read_csv(path)
df = df.sort_values(df.columns[0], ascending=True)
df